---

## ToolRuntime

도구는 에이전트 상태, 런타임 컨텍스트 및 장기 메모리에 액세스할 수 있을 때 가장 강력합니다. 이를 통해 도구는 컨텍스트 인식 결정을 내리고, 응답을 개인화하며, 대화 전반에 걸쳐 정보를 유지할 수 있습니다.

`ToolRuntime` 매개변수를 통해 다음 런타임 정보에 액세스할 수 있습니다:

| 속성 | 설명 |
|:---|:---|
| **state** | 실행을 통해 흐르는 변경 가능한 데이터 (메시지, 카운터, 커스텀 필드) |
| **context** | 사용자 ID, 세션 세부 정보 등 불변 구성 정보 |
| **store** | 대화 전반에 걸친 영구 장기 메모리 |
| **stream_writer** | 도구 실행 중 커스텀 업데이트 스트리밍 |
| **config** | 실행을 위한 RunnableConfig |
| **tool_call_id** | 현재 도구 호출의 고유 ID |

---
### Stream Writer

`runtime.stream_writer`를 사용하면 도구 실행 중 커스텀 업데이트를 스트리밍할 수 있습니다. 이는 장시간 실행되는 도구에서 사용자에게 진행 상황을 실시간으로 알려줄 때 유용합니다.

스트리밍된 업데이트는 `stream_mode="custom"`으로 수신할 수 있습니다. Stream Writer는 LangGraph 실행 컨텍스트 내에서만 사용할 수 있습니다.

아래 코드는 Stream Writer를 사용하여 진행 상황을 스트리밍하는 도구 예시입니다.



### 💡 한눈에 보는 비교표

| 모드 (`stream_mode`) | 스트리밍 시점 | 반환하는 데이터 | 주요 특징 및 보안 관점 |
| --- | --- | --- | --- |
| **`values`** | 각 노드 실행 **완료 시** | 그래프의 **전체 상태(State) 스냅샷** | 매번 모든 데이터가 노출되므로 보안 및 메모리 오버헤드 주의 |
| **`updates`** | 각 노드 실행 **완료 시** | 해당 노드가 **변경/추가한 상태(Delta)** | 변경점만 추적하므로 효율적. 가장 표준적으로 사용됨 |
| **`custom`** | 노드 실행 **중간 (실시간)** | 개발자가 **직접 정의한 이벤트 데이터** | State를 철저히 은닉하고 안전한 데이터만 선별 전송 가능 |


In [2]:
from feature.tools.ToolRuntimeStreamWriter import ToolRuntimeAgent

workflow = ToolRuntimeAgent()

In [3]:
inputs = {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]}

for chunk in workflow.graph.stream(inputs, stream_mode="custom"):
    if "progress" in chunk:
        # 진행률 계산 및 출력
        percentage = (chunk["progress"] / chunk["total"]) * 100
        print(f"Progress: {percentage:.0f}%")
    elif "stage" in chunk and chunk["stage"] == "completed":
        # 완료 메시지 출력
        print(f"Completed processing {chunk['messages']}")
    else:
        print(chunk)


Progress: 20%
Progress: 40%
Progress: 60%
Progress: 80%
Progress: 100%
Completed processing 50
Completed processing In the 서울
Acquired data for city: 서울
